In [ ]:
import os
import string
from datasets import load_dataset
from mlx_tune import FastLanguageModel, SFTTrainer, SFTConfig
from transformers import TrainingArguments, set_seed
from mlx_lm import generate
from mlx_tune.chat_templates import get_chat_template, standardize_sharegpt

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "google/gemma-3-270m",
    max_seq_length = 2048,
    load_in_4bit = False
)

In [ ]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

In [ ]:
try:
    from gemmaiku import get_syllable_count_for_line as count_syllables_line
except ModuleNotFoundError:
    import sys
    import os
    sys.path.append(os.path.abspath("../../src"))
    from gemmaiku import get_syllable_count_for_line as count_syllables_line


In [ ]:
def evaluate_haiku(text):
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if len(lines) != 3:
        return [0, 0, 0], False
    counts = [count_syllables_line(l) for l in lines]
    is_perfect = (counts == [5, 7, 5])
    return counts, is_perfect

In [ ]:
message = [
             {"role": "user", "content": "I want to become a hardware engineer"}
        ]

In [ ]:
prompt = tokenizer.apply_chat_template(
            message,
            tokenize=False,
            add_generation_prompt=True
        )

In [ ]:
response = generate(model, tokenizer, prompt=prompt, max_tokens=64)

In [ ]:
print(response)

In [ ]:
haiku = response.split("<end_of_turn>")[0].strip()

In [ ]:
counts, is_perfect = evaluate_haiku(haiku)

In [ ]:
total_syllables = sum(counts)

In [ ]:
print(total_syllables)